# WhisperWatch ensemble experiment

Compare boosted trees, Whisper confidence, and a blend at an editor review budget of **20%**. Use the hosted **CPU** runtime in Google Colab. Run the cells in order. Have `whisperwatch-ensemble-speaker-split-code.zip` and the original `whisperwatch-boosted-results.zip` ready.

The earlier full-development comparison is exploratory. We use a fixed split that assigns about half of the development speakers to tuning and half to verification. Every clip from a speaker stays in the same partition, so clip counts and class proportions can differ. Both partitions were part of the earlier full-development analysis, so verification is an internal check. The official test split is outside this notebook.

## 1. Install dependencies

In [ ]:
%pip install -q scikit-learn==1.8.0 pandas==2.2.3 "pyarrow>=23.0.1" joblib==1.5.3

## 2. Load the source and the saved boosted predictions

Upload only the two named ZIPs when prompted. This setup reads the prediction CSV from the results ZIP. It does not load the model file inside it.

In [ ]:
import sys
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

PROJECT = Path("/content/whisperwatch_ensemble_experiment")
SOURCE_FILES = (
    "src/__init__.py", "src/features.py", "src/inference.py",
    "src/train.py", "src/ensemble_experiment.py",
)

def upload_zip(prompt):
    print(prompt)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Choose one ZIP for this step.")
    return BytesIO(next(iter(uploaded.values())))

with ZipFile(upload_zip("Choose whisperwatch-ensemble-speaker-split-code.zip")) as source:
    missing = set(SOURCE_FILES) - set(source.namelist())
    if missing:
        raise ValueError(f"The code ZIP is missing {sorted(missing)}.")
    for filename in SOURCE_FILES:
        path = PROJECT / filename
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(source.read(filename))

sys.path.insert(0, str(PROJECT))
from src.ensemble_experiment import read_development

dev = read_development(upload_zip("Choose whisperwatch-boosted-results.zip"))
print(f"Loaded {len(dev):,} English development clips.")

## 3. Count overlap and compare rankings

On the full development set, count how many actual high-error clips land in both queues, only the trees queue, only the Whisper queue, or neither.

For the controlled comparison, read archived speaker IDs by clip ID to assign the two partitions. Convert both signals to percentile ranks within each partition. A higher percentile means a riskier clip. Use tuning to choose among 25%, 50%, and 75% boosted-tree weight. Whisper gets the remaining weight. Then use that single weight on verification. Tuning selects the weight that catches the most high-error clips, preferring 50%, then 25%, then 75% for ties. Clip ID breaks ranking ties. Missing Whisper confidence goes to the front, matching the original baseline.

Review exactly the top 20% of each partition, rounded up. Precision means the share of reviewed clips that actually had high errors. Recall means the share of all high-error clips that made it into the review queue. Speaker IDs determine the split. Labels select the weight on tuning and measure results. The ranking uses the two model signals.

In [ ]:
import pandas as pd
from IPython.display import display
from src.ensemble_experiment import compare_ensemble, detected_overlap

overlap = pd.Series(detected_overlap(dev), name="count")
display(overlap.to_frame())

weight, tuning, verification, tuning_results, verification_results = compare_ensemble(dev)
print(f"Tuning: {len(tuning):,} clips")
display(tuning_results.round(4))
print(f"Verification: {len(verification):,} clips. Selected tree weight: {weight:.0%}")
display(verification_results.round(4))

standalone = verification_results.loc[["boosted_trees", "whisper_confidence"], "caught"]
best_standalone = int(standalone.max())
blend_caught = int(verification_results.at["selected_blend", "caught"])
blend_improved = blend_caught > best_standalone
if blend_improved:
    print(f"The blend found {blend_caught - best_standalone} more high-error clips than the strongest standalone method.")
    print("Keep the blend as the candidate for a frozen test run. Treat small gains as provisional.")
else:
    print("The blend did not beat the strongest standalone method on verification.")
    print("Run the feature experiment in step 4.")

## 4. Check three additional features if the blend falls short

The candidate adds the share of repeated words, the share of repeated adjacent word pairs, and average word length. These come from Whisper's generated transcript. They may help detect repeated phrases or unusually short or long words. The candidate uses the same full English training split and tree settings as the saved boosted model.

This experiment runs if the blend falls short. It scores the added features on the tuning clips at the same 20% review budget. Treat its result as exploratory because the blend result prompted the follow-up.

In [ ]:
from src.ensemble_experiment import compare_added_features

feature_results = None
recommendation = "Freeze the selected blend as the candidate for a future official test."
if blend_improved:
    print(recommendation)
else:
    feature_results = compare_added_features(tuning)
    display(feature_results.round(4))
    best_tuning_baseline = int(feature_results.loc[standalone.index, "caught"].max())
    feature_gain = int(feature_results.at["boosted_with_added_features", "caught"]) - best_tuning_baseline
    if feature_gain > 0:
        recommendation = (
            f"The extra features caught {feature_gain} more high-error clips on tuning. "
            "Keep the feature model as an exploratory candidate and confirm the gain on fresh data."
        )
    else:
        recommendation = (
            f"Keep {standalone.idxmax()} as the candidate based on verification. "
            "The added features fell short of a tuning improvement."
        )
    print(recommendation)

## 5. Download the overlap, comparison tables, and split assignments

In [ ]:
import json
from zipfile import ZIP_DEFLATED
from src.ensemble_experiment import BOOSTED_WEIGHTS, REVIEW_FRACTION, SPLIT_SEED

results_zip = PROJECT.parent / "whisperwatch-ensemble-speaker-split-results.zip"
with ZipFile(results_zip, "w", compression=ZIP_DEFLATED) as results:
    results.writestr("development_overlap.csv", overlap.to_csv())
    results.writestr("tuning_results.csv", tuning_results.to_csv())
    results.writestr("verification_results.csv", verification_results.to_csv())
    split = pd.concat([
        pd.Series("tuning", index=tuning.index),
        pd.Series("verification", index=verification.index),
    ]).sort_index().rename("partition")
    results.writestr("development_split.csv", split.to_csv())
    if feature_results is not None:
        results.writestr("added_features_tuning.csv", feature_results.to_csv())
    results.writestr("protocol_and_recommendation.json", json.dumps({
        "review_fraction": REVIEW_FRACTION,
        "split_seed": SPLIT_SEED,
        "split_group": "speaker_id",
        "predefined_tree_weights": BOOSTED_WEIGHTS,
        "selected_tree_weight": weight,
        "recommendation": recommendation,
    }, indent=2))

files.download(str(results_zip))